## מנוסחה לקוד: תהליך התרגום ושגיאות נפוצות

לתרגם משוואה פיזיקלית לקוד זו לא רק "העתקת סימנים" — יש לזה תהליך שיטתי, שאפשר וכדאי להפעיל בכל פעם מחדש. בסעיף הזה נעבור על התהליך הזה צעד-צעד על דוגמה מלאה, ואז נבנה יחד "מאגר באגים": שלושת סוגי הטעויות הנפוצים ביותר כשהתהליך משתבש. באגים מהסוג הזה מסוכנים במיוחד, כי הקוד **רץ בלי שגיאה** — הוא פשוט מחזיר תוצאה פיזיקלית לא נכונה, בלי שום התראה.

### תהליך שיטתי: מנוסחה לביטוי ב-NumPy

בכל פעם שמתרגמים משוואה פיזיקלית לקוד, שווה לעבור על ארבעה שלבים, בסדר הזה:

1. **זיהוי כל גורם במשוואה** — האם הוא סקלר קבוע (מספר בודד, כמו מהירות התחלתית אחת), מערך (למשל טווח זמנים, או תוצאה של כמה ריצות), או קבוע פיזיקלי (`scipy.constants`)?
2. **החלטה על סדר הפעולות והביטוי המתאים ב-NumPy** — אילו סוגריים נדרשים כדי שסדר החישוב יתאים לנוסחה? אילו פעולות פועלות איבר-איבר (סעיף 7.7), ואילו הן סקלר מול מערך?
3. **כתיבת הביטוי שורה-שורה** — לא לנסות לדחוס את כל הנוסחה לשורה אחת מסובכת בניסיון הראשון. לבנות בהדרגה, ולהדפיס תוצאות ביניים כדי לוודא שכל שלב עשה מה שהתכוונתם.
4. **בדיקת סבירות בסוף** — יחידות (כל הגדלים עדיין ב-SI?), סדר גודל (התוצאה הגיונית פיזיקלית?), ומקרי קצה (מה קורה בערכי קיצון, כמו זווית של $0°$ או $90°$?).

### דוגמה: מהנוסחה לקוד — טווח הזריקה

נתרגם את נוסחת הטווח האופקי של זריקה:

$$R(v_0,\theta) = \frac{v_0^2 \sin(2\theta)}{g}$$

**שלב 1 — זיהוי גורמים:** $v_0$ ו-$\theta$ הם סקלרים (מהירות שיגור וזווית שיגור בודדות), $g$ הוא קבוע פיזיקלי. הזווית $\theta$ נתונה במעלות — ולכן, כמו בכל שימוש בפונקציה טריגונומטרית של NumPy (סעיף 7.7), חייבים להמיר אותה לרדיאנים **לפני** שמכניסים אותה ל-`np.sin`.

**שלב 2 — סדר הפעולות:** קודם `2*theta` (בתוך ה-`sin`), אז `np.sin(...)`, אז `v0**2 * ...`, ולבסוף חילוק ב-`g`. כל צעד כזה יהפוך לשורת קוד נפרדת.

In [ ]:
import numpy as np
from scipy import constants

g = constants.g    # m/s^2
v0 = 20.0           # m/s
theta_deg = 45      # מעלות

# שלב 3: כתיבה שורה-שורה
theta = np.radians(theta_deg)          # המרה לרדיאנים לפני sin
numerator = v0**2 * np.sin(2 * theta)
R = numerator / g

print(f"R = {R:.2f} m")

# שלב 4: בדיקת סבירות - מקרי קצה
print("R בזווית 0°:", round(v0**2 * np.sin(2 * np.radians(0)) / g, 6))    # ציפייה: 0
print("R בזווית 90°:", round(v0**2 * np.sin(2 * np.radians(90)) / g, 6))  # ציפייה: 0

התוצאה, כ-40.8 מטר עבור $v_0=20$ מ/ש בזווית $45°$, נמצאת בסדר גודל סביר לזריקת כדור ביד. בדיקת מקרי הקצה מאששת את הפיזיקה: בזווית $0°$ הכדור לא עף אופקית בכלל (הטווח 0), ובזווית $90°$ הוא עולה ויורד באותה נקודה (הטווח גם כן 0) — בדיוק כפי שמצפים ממשוואת הטווח.

### בדקו את עצמכם

In [ ]:
from jupyterquiz import display_quiz

questions = [
    {
        "question": "לפי התהליך שראינו, מה השלב הראשון בתרגום משוואה לקוד — עוד לפני כתיבת שורת קוד אחת?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "לכתוב את כל הביטוי בשורה אחת ואז לתקן שגיאות אם יש", "correct": False, "feedback": "לא — זה בדיוק ההפך מהתהליך המומלץ; עדיף לבנות בהדרגה."},
            {"answer": "לזהות כל גורם במשוואה: האם הוא סקלר קבוע, מערך, או קבוע פיזיקלי", "correct": True, "feedback": "נכון."},
            {"answer": "לבדוק את יחידות המידה של התוצאה הסופית", "correct": False, "feedback": "זה שלב חשוב — אבל הוא שלב 4, בדיקת הסבירות בסוף, לא הראשון."},
            {"answer": "לחפש את הנוסחה המדויקת בספר לימוד", "correct": False, "feedback": "לא קשור לתהליך התרגום עצמו — התהליך מתחיל מרגע שהנוסחה כבר בידיכם."}
        ]
    },
    {
        "question": "בדוגמת הטווח, למה חובה להפעיל את <code>np.radians</code> על הזווית לפני <code>np.sin</code>, ולא בשלב כלשהו אחרי?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "כי np.radians משנה גם את הערך של g", "correct": False, "feedback": "לא — np.radians פועל רק על הזווית שהועברה לו, ולא נוגע במשתנים אחרים."},
            {"answer": "זה לא באמת משנה — אפשר להמיר בכל שלב בתהליך, גם אחרי החישוב", "correct": False, "feedback": "לא — זה בדיוק ההפך; ראינו בסעיף 7.7 שהמרה בשלב הלא נכון נותנת תוצאה פיזיקלית שגויה."},
            {"answer": "כי np.sin לא מקבל בכלל משתנים מטיפוס float", "correct": False, "feedback": "לא נכון — np.sin מקבל בדיוק מספרים מטיפוס float; הבעיה היא ביחידה, לא בטיפוס."},
            {"answer": "כי np.sin מצפה שהערך שהיא מקבלת כבר יהיה ברדיאנים; קלט במעלות מחושב כאילו הוא רדיאנים, וזו זווית אחרת לגמרי", "correct": True, "feedback": "נכון."}
        ]
    }
]

display_quiz(questions)

### נסו בעצמכם

הפעילו את התהליך בעצמכם על נוסחת **הגובה המרבי** של הזריקה:

$$y_{max}(v_0,\theta) = \frac{(v_0\sin\theta)^2}{2g}$$

עברו על ארבעת השלבים: אילו גורמים יש כאן, מה סדר הפעולות הנכון (שימו לב: הריבוע חל על **התוצאה** של $v_0\sin\theta$, לא על $v_0$ בלבד!), כתבו את הביטוי שורה-שורה, ובדקו סבירות עבור $v_0=20$ מ/ש ו-$\theta=45°$.

In [ ]:
# v0 = 20.0
# theta_deg = 45
#
# שלב 1: מהם הגורמים כאן? (סקלר קבוע / קבוע פיזיקלי)
# שלב 2-3: כתבו כאן, שורה-שורה
# y_max = ...
#
# שלב 4: בדיקת סבירות
# print(y_max)

`````{admonition} פתרון
:class: dropdown, tip
```python
v0 = 20.0
theta_deg = 45

theta = np.radians(theta_deg)               # שלב 1+2: המרה לרדיאנים לפני sin
y_max = (v0 * np.sin(theta))**2 / (2 * g)   # שלב 3: הריבוע חל על התוצאה של v0*sin(theta)

print(f"y_max = {y_max:.2f} m")             # שלב 4: כ-10.2 מ' — סביר, וכרבע מגודל הטווח שחישבנו למעלה
```
`````

עכשיו, כשראינו איך התהליך אמור להיראות כשהוא עובד נכון — נעבור למאגר הבאגים: מה קורה, בשקט, כשכל אחד מהשלבים האלה משתבש.

### באג 1: קדימות פעולות

הנוסחה לתקופת מטוטלת פשוטה היא $T = 2\pi\sqrt{L/g}$. הקוד הבא ניסה לממש אותה — אבל התוצאה שגויה. מצאו את הבאג לפני שתגלו את הפתרון:

In [1]:
import numpy as np
from scipy import constants

L = 1.0   # מטר
T_buggy = 2 * np.pi * np.sqrt(L) / constants.g   # מה פה לא בסדר?
print(T_buggy)

0.6407065926875729


`````{admonition} פתרון
:class: dropdown, tip
הבאג: `np.sqrt(L) / constants.g` מחשב $\sqrt{L}/g$, לא $\sqrt{L/g}$. כדי לחשב את השורש של המנה כולה, צריך את ה-`g` **בתוך** ה-`sqrt`:

```python
T_fixed = 2 * np.pi * np.sqrt(L / constants.g)
print(T_fixed)
```
`````

### באג 2: חילוק שלמים

רוצים לחשב את מהירות הממוצע של כדור שעבר מרחק 7 מטרים ב-2 שניות ($v=d/t$):

In [2]:
d = 7   # מטר
t = 2   # שניות
v_buggy = d // t   # מה פה לא בסדר?
print(v_buggy)

3


`````{admonition} פתרון
:class: dropdown, tip
הבאג: `//` הוא חילוק שלמים (מחזיר את החלק השלם, מעוגל כלפי מטה) — `7 // 2` נותן `3`, לא `3.5`. עבור גדלים פיזיקליים רציפים צריך את אופרטור החילוק הרגיל `/`:

```python
v_fixed = d / t
print(v_fixed)   # 3.5
```

חילוק שלמים שגוי מסוכן במיוחד כי הוא לא זורק שגיאה, ולעיתים אף נותן תוצאה "סבירה למראה" (כמו 3, לא רחוק מ-3.5).
`````

### באג 3: יחידות

מודדים זווית שיגור של $30°$ ורוצים את רכיב המהירות האנכי, $v_y = v_0 \sin\theta$:

In [3]:
v0 = 20.0
theta_deg = 30
vy_buggy = v0 * np.sin(theta_deg)   # מה פה לא בסדר?
print(vy_buggy)

-19.760632481857236


`````{admonition} פתרון
:class: dropdown, tip
הבאג: `np.sin` מצפה לזווית ברדיאנים, ו-`theta_deg` הוא במעלות. חייבים להמיר קודם עם `np.radians`:

```python
vy_fixed = v0 * np.sin(np.radians(theta_deg))
print(vy_fixed)   # 10.0, כצפוי מ- v0 * sin(30°) = v0 * 0.5
```
`````

### נסו בעצמכם

הקוד הבא מנסה לחשב את האנרגיה הפוטנציאלית הממוצעת של שלושה גופים, $\bar{E} = mg\bar{h}$, כאשר `heights` הוא מערך גבהים בס"מ (לא במטר!). יש כאן שילוב של שני סוגי הבאגים שראינו. אתרו ותקנו:

In [4]:
m = 2.0  # ק"ג
heights_cm = np.array([150, 200, 80])   # ס"מ

# avg_energy_buggy = m * constants.g * heights_cm.mean()
# print(avg_energy_buggy)

# תקנו כאן:
# avg_energy_fixed = ...


`````{admonition} פתרון
:class: dropdown, tip
```python
# הבאג: heights_cm הוא בסנטימטרים, ו-g בנוסחה מצפה למטרים.
heights_m = heights_cm / 100
avg_energy_fixed = m * constants.g * heights_m.mean()
print(avg_energy_fixed)
```
`````

### בדקו את עצמכם

In [ ]:
from jupyterquiz import display_quiz

questions = [
    {
        "question": "מה התוצאה של <code>9 // 2</code> בפייתון?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "4.5", "correct": False, "feedback": "זו הייתה התוצאה של / (חילוק רגיל), לא של // (חילוק שלמים)."},
            {"answer": "5", "correct": False, "feedback": "לא — // מעגל כלפי מטה, לא כלפי מעלה."},
            {"answer": "-4", "correct": False, "feedback": "לא — שני המספרים חיוביים, אז אין סיבה לתוצאה שלילית."},
            {"answer": "4", "correct": True, "feedback": "נכון: // מעגל כלפי מטה לשלם הקרוב."}
        ]
    },
    {
        "question": "מדוע באג של קדימות פעולות (כמו sqrt(L)/g במקום sqrt(L/g)) מסוכן במיוחד?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "כי הקוד רץ בהצלחה ומחזיר מספר, אבל הוא פיזיקלית שגוי", "correct": True, "feedback": "נכון. זו בדיוק הסיבה שקשה לתפוס באגים כאלה — אין הודעת שגיאה שמצביעה עליהם."},
            {"answer": "כי פייתון זורק שגיאת תחביר", "correct": False, "feedback": "לא — התחביר תקין לגמרי, הקוד רץ בלי בעיה."},
            {"answer": "כי NumPy לא תומך בחילוק בתוך sqrt", "correct": False, "feedback": "לא — NumPy תומך בכך היטב, השאלה היא רק סדר הפעולות שכתבתם."},
            {"answer": "כי הוא גורם ל-Python לקרוס לגמרי", "correct": False, "feedback": "לא — הקוד ממשיך לרוץ כרגיל; זו בדיוק הבעיה, שאין שום קריסה שתסמן את הטעות."}
        ]
    }
]

display_quiz(questions)